# Finding transients and broker info in FASTDB

This example assumes you want to find things that were passed by a given broker filter, and you want to inspect the information that the broker gave us about those objects.

In [1]:
# First, import the things and make our FASTDB connection:

from pprint import pp

import pandas

from fastdb_client import FASTDBClient
fdb = FASTDBClient( 'production' )


## Getting sources from a broker.

First, a reminder of LSST terminology.  A *diaObject* is a single transient or variable.  ("dia" = "differential imaging analysis").  They are identified by LSST by doing image subtraction and scanning the difference image for left over spots.  Any identified detections are called *diaSources*.  A *diaSource* is either associated with an existing *diaObject*, or, if there aren't any existing ones, a new *diaObject* is created.  Shortly after the detecting, LSST will send an alert to brokers for this diaSource.  That alert will include all the previous diaSources from the same diaObject.  It will also include any forced photometry, as *diaForcedSource* records.  (LSST performs forced photometry for identified diaObjects on ... all? ... of the images that include its position.  This happens at some delay, so generally you'll only get forced photometry up to a day, or a few days, before the time the alert is sent out.)

Brokers ingest alerts, perhaps run some code on them, filter them, and then produce their own messages that FASTDB then ingests.  What we get from brokers is heterogeneous.  *Some* of the brokers include everything that was in the original LSST alert, but not all of them do.  Brokers add different information themselves.  (They also use different mechanisms for dispersing the information.)  All of these things together mean that it's a bit complicated for FASTDB to listen to multiple brokers, so right now we're just listening to a couple of them.  Our primary brokers are Fink and Pitt-Google, with, as of this writing, Fink providing the most additional information beyond the original LSST alert.

For the first example, we're going to get the diasources that were produces by given broker in a given time period.  This will *not* give us full lightcurves, it will only give us information about the individual sources that the brokers sent out on a filter.

But, first, let's get the list of brokers and topics that we know about:


In [2]:
brokersandtopics = fdb.post( "ltcv/knownbrokertopics/realtime" )
# Sort by broker and topic.  It may be like this anyway, but just in case
brokersandtopics.sort( key=lambda x: ( x[0], x[1] ) )
print( "Known brokers and topics" )
curbroker = None
for bt in brokersandtopics:
    if bt[0] != curbroker:
        curbroker = bt[0]
        print( curbroker )
    print( f"    {bt[1]}" )


Known brokers and topics
Fink
    fink_most_likely_sn_lsst
    fink_remove_unlikely_transients_lsst
    fink_sn_near_galaxy_candidate_lsst
    fink_uniform_sample_lsst
    ftransfer_lsst_2026-03-20_872471
Pitt-Google
    lsst-alerts


Let's pull all sources from the last three days for the `fink_most_likely_sn_lsst` topic.  (Note that this topic went live mid-June, so there will only be sources from around then.)  As will usually be the case for alerts, we're using the `realtime` processing version.  (Normally, if you ask for things detected in the last n days, that's in the days previous to the time at which you make the query.  You can specify `mjd_now` to tell the server to treat the current time as as different time; that's useful for trying to go back and redo stuff, or for examples like this.)  The time of an alert, in thise case, is defined as the MJD of the source detection, *not* the time that we received the alert.



In [3]:
objs = fdb.post( "ltcv/sourcesfrombroker/Fink/fink_most_likely_sn_lsst/realtime", 
                    json={ 'detected_in_last_days': 7, 'mjd_now': 61207.5 } )
print( f"Found {len(objs)} different transients from the fink_most_likely_sn_lsst topic in the 7 days before 61207.5" )

Found 18689 different transients from the fink_most_likely_sn_lsst topic in the 7 days before 61207.5


What you get back is a dict.  The key of the dict is the object's rootid.  Each value is itself a dict, with keys `diaobjectids` and `diasources`.  `diaobjectids` is just a list of the `diaobjectids` associated with this `rootid` from the alerts we found; this will *usually*, but not always, be a single-element list.  `diasources` is itself a dictionary with keys `diasourceid`, `visit`, `band`, `mjd`, `flux`, `fluxerr`, `ra`, `dec`, `raerr`, `decerr`, `ra_dec_cov`, and `info` (and a few others).  Each one of those keys is a list.

OK, that was complicated.  Basically, you get a dictionary keyed by object (identified by the rootid), because it's possible that the broker will have identified more than one source for the same object on this alert stream in the time period.  For each object, you get a bunch of lists; the length of those lists will be the number of sources that we got alerts about from that broker for that object in the time perioud you specified.

So, cherry-picking an example I know has several sources:

In [4]:
print( f"rootid 5c7434f9-127f-44d6-a812-29ac3bfa6c85 had len(objs['5c7434f9-127f-44d6-a812-29ac3bfa6c85']['diasources']['diasourceid'])\n"
       f"diasources that triggered an alert.")

# Let's make a table of the basic info

subset = { k: v for k,v in objs['5c7434f9-127f-44d6-a812-29ac3bfa6c85']['diasources'].items()
           if k in [ 'diasourceid', 'mjd', 'visit', 'band', 'ra', 'dec', 'flux', 'fluxerr' ] }
df = pandas.DataFrame( subset )
df


rootid 5c7434f9-127f-44d6-a812-29ac3bfa6c85 had len(objs['5c7434f9-127f-44d6-a812-29ac3bfa6c85']['diasources']['diasourceid'])
diasources that triggered an alert.


,diasourceid,visit,band,ra,dec,mjd,flux,fluxerr
0,170534263268049107,2026061300073,u,150.477263,0.757265,61204.975954,-12489.458,1017.19415
1,170534263536484732,2026061300075,u,150.477325,0.757171,61204.977249,-11452.573,1023.78600
2,170534263671750725,2026061300076,u,150.477321,0.757187,61204.977906,-12333.346,992.41570
3,170534264133124174,2026061300079,g,150.477322,0.757158,61204.981235,-8888.522,462.20682
4,170534264267866188,2026061300080,g,150.477338,0.757193,61204.981752,-8453.861,455.65110


All these fluxes are negative!  I *think* this is because LSST detects both positive and negative outliers on their difference images, so this is something that was dimmer in all of the search images as compared to the referecene template image they were using for image subtraction.

For each one of these sources, there is also a packet of information that the broker gave us, in the `info` list.  What information is there will be different for different brokers, and even possibly different for different topics from the same broker.

In [5]:
objsourceinfo = objs['5c7434f9-127f-44d6-a812-29ac3bfa6c85']['diasources']
for i in range( len( objsourceinfo['diasourceid'] ) ):
    print( "\n\n========================================================\n" )
    print( f"Broker info for diasource {objsourceinfo['diasourceid'][i]}:" )
    pp( objsourceinfo['info'][i] )
    




Broker info for diasource 170534263268049107:
{'xm': {'tns_type': None,
        'vsx_Type': None,
        'gcvs_type': None,
        'x3hsp_type': None,
        'x4lac_type': None,
        'gaiadr3_Plx': -0.20800000429153442,
        'spicy_class': None,
        'simbad_otype': 'QSO',
        'tns_fullname': None,
        'tns_redshift': None,
        'gaiadr3_e_Plx': 0.19059999287128448,
        'gaiadr3_DR3Name': 'Gaia DR3 3833660595497395968',
        'gaiadr3_VarFlag': 0,
        'legacydr8_fqual': 1,
        'legacydr8_pstar': 0.0010000000474974513,
        'legacydr8_zphot': 1.184000015258789,
        'legacydr8_e_zphot': 0.41499999165534973,
        'mangrove_ang_dist': None,
        'mangrove_lum_dist': None,
        'mangrove_2MASS_name': None,
        'mangrove_HyperLEDA_name': None},
 'clf': {'cats_class': 11,
         'cats_score': 0.9987855553627014,
         'earlySNIa_score': -1.0,
         'snnSnVsOthers_score': 0.7324028611183167,
         'elephant_kstest_science':